In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker 
import pickle
import re, sys
import pymongo
import sncosmo
from scipy.stats.distributions import chi2
from astropy.table import Table


In [ ]:
# Depending on whether the library was added or not, you might have to manually add the base dir to the python path
sys.path.append('/Users/jnordin/github/ampelFeb25')

In [ ]:
# We try to load the main loader class for warped templates
from warpTemplate import WarpfitTemplateLoader

What happens here:
1. Pick a SN from BTS together with the base class you which to use to fit against
2. Retrieve SN information: redshift and photometry.
3. Decide on how many templates you wish to fit to and (randmoly reterieve these). Can become many!
4. Fit SN to each of the selected templates
5. Print/plot results for the best fit

### 1. Pick a SN from BTS together with the base class you which to use to fit against

In [ ]:
#snname = 'ZTF20abbbumr'
#snname = 'ZTF19aailltc'

# Bad cases from suleyman
#snname = 'ZTF19aanijpu' # Ic
#template_class_id = 8   # Ok, but not amazing w/o itself
#template_class_id = 4   # The same ...

#snname = 'ZTF19abcegvm' # Ic
#template_class_id = 8   # Ok, but not amazing
#template_class_id = 4   # The same ...

snname = 'ZTF18abfcmjw' # Ib
#template_class_id = 0   # Ok, but not amazing
template_class_id = 4   # The same ...



cutself = True # If set to False, it typically should fit perfectly



In [ ]:
warpclasses = [
    'SN Ib', 'SN Ia-91bg', 'SN II', 'SN Ia-91T',
    'SN Ib/c', 'SN Ia-pec', 'SN IIP', 'SN Iax', 'SN Ic',
    'SLSN-II']

In [ ]:
print('Fitting to templates of class', warpclasses[template_class_id])

### 2. Retrieve SN information: redshift and photometry.

In [ ]:
df_bts = pd.read_csv('/Users/jnordin/data/ztf/bts/bts_explorer_241122.csv')

In [ ]:
sndict = df_bts.loc[df_bts['ZTFID']==snname,:].iloc[0]

In [ ]:
print('looking at {} - type {}'.format(snname, sndict['type']))

In [ ]:
client = pymongo.MongoClient()
db = client.bts_ipacfp_strictbase    # Only final lc. try bts_ipacfp_strictbase_full for all alerts

In [ ]:
def get_db_table( name, database, tabulators ):
    """
    For ZTF name, get photopoints and then tables. 
    """
    from ampel.ztf.view.ZTFFPTabulator import ZTFFPTabulator
    from ampel.ztf.util.ZTFIdMapper import ZTFIdMapper

    # Name
    if isinstance(name, int):    
        # Assuming this is already a DB stock
        stock = int
    elif re.search('ZTF', name):
        stock = ZTFIdMapper.to_ampel_id(name)
    else:
        raise ValueError(f"Cannot parse {name}" )

    # Obtain photopoints
    dps = [dp for dp in database.t0.find({'stock':stock})]

    # Convert to table(s)
    ftables = [
        tabulator.get_flux_table(dps) for tabulator in tabulators
    ]
    if len(ftables)>1:
        raise NotImplementedError("Debug appending two tabulator tables.")
    return ftables.pop(0)
def get_ztftable_from_ampel(ztfid: str, dbhandle, include_sigma: float = 5., **kwarg) -> Table:
    """
    Given a ZTF name and a local AMPEL DB:
    - Retrieve available photometric data.
    - Reject outliers. 
    - Return an astropy table useful e.g. for sncosmo. 

    Parameters:
    - ztfid: str : ZTF name, e.g. ZTF18aaayemw
    - dbhandle: Database : AMPEL MongoDB handle
    - inclusion_sigma: float : Sigma threshold for outlier rejection.
    - kwarg: dict : Additional arguments added to table meta.

    tab = get_ztftable_from_ampel('ZTF22aaa', dbhandle, inclusion_sigma=5., z=0.03, type='SN Ia')

    """
    from ampel.ztf.view.ZTFFPTabulator import ZTFFPTabulator
    from ampel.ztf.util.ZTFIdMapper import ZTFIdMapper

    # Load photopoints from AMPEL DB
    tabulators = [
        ZTFFPTabulator(inclusion_sigma=include_sigma)
        ]
    tab = get_db_table(ztfid, database=dbhandle, tabulators=tabulators)
    tab.sort('time')

    tab.meta = {
            'object_id':ztfid,
            **kwarg
        }

    return tab

In [ ]:
# Lets grab some data, starting with the target SN
# Warning: Note that the redshift and type also gets added to the meta information of the photometry table!
tab = get_ztftable_from_ampel( snname, db, 
                              redshift=float(sndict['redshift']), 
                              type = sndict['type'] )

In [ ]:
tab

In [ ]:
# Estimate significant detection
dps = sum( (np.abs(tab['flux']) / tab['fluxerr'])>5 )

In [ ]:
dps

### 3. Decide on how many templates you wish to fit to and (randmoly reterieve these).

In [ ]:
# Parameters for fit template retrieval

if cutself:
    exclude_input = [snname] # Will reject any warptemplate containing any of these (either as sn or template basis)
else:
    exclude_input = []
# How to define templates?
# - How many templates per sn basis? 
#      * if 'all' it will return one copy of each template, 
#.     * if int it will return that many, drawn according to the template probability, 
#      * if -int it will return that many copies drawn from a uniform probabilitiy
#      Note: draws made with replacement, so multiple copies can be returned if int is larger than the available number of templates (often 3)
template_selection = 'all'
# - How many sn basis?
#.     * if 'all', take one of each
#.     * if an int, draw these randomly (with replacement)
#.     Note: how many templates are returned is decided by the above parameter.
snbasis_selection = 'all'
warpdir = '/Users/jnordin/data/models/sncosmo/warpmod'

In [ ]:
warploader = WarpfitTemplateLoader(warpdir)

In [ ]:
templates = warploader.get_templates(
    fitclass=warpclasses[template_class_id],
    exclude_input = exclude_input, 
    template_selection=template_selection,
    snbasis_selection=snbasis_selection,
    random_seed=42
)

### 4. Fit SN to each of the selected templates

In [ ]:
# Loop through templates, fit and plot the best...
bestmodel = None
for k, template in enumerate(templates):
    print(k,len(templates),template)
    fitprop = ['t0', 'amplitude', 'hostebv']
    try:
        wresult, wfitted_model = sncosmo.fit_lc(
                        tab, template['model'],
                        fitprop,  # parameters of model to vary
                    )
    except RuntimeError:
        print('fit failed')
        continue
    # Try to limit to models that use all the data
    if wresult['ndof']<(dps-len(fitprop)):
        print('... datapoints rejected, skipping this for now.')
        continue
    
    # Evaluate
    if wresult['success']:
        if bestmodel is None:
            bestmodel = [template,wfitted_model, wresult['chisq'] / wresult['ndof']]
        elif wresult['chisq'] / wresult['ndof'] < bestmodel[2]:
            bestmodel = [template,wfitted_model, wresult['chisq'] / wresult['ndof']]
#    sncosmo.plot_lc(tab, wfitted_model)

### 5. Print/plot results for the best fit

In [ ]:
bestmodel[1].minwave()

In [ ]:
_ = sncosmo.plot_lc(tab, bestmodel[1])